# Chr19 Amplicon – PFI Association Analysis

This notebook tests whether the **chr19 amplicon as a whole** — rather than any single gene — is associated with PFI category (short / medium / long).

Author: Franziska Niemeyer

In [ ]:
ADATA_PATH = "../../../quality_control/primary-cohort/adata.h5ad"

CNV_FILES = {
    "BST2 amp"  : "../outputs/BST2_results/BST2_amplification_spots_annotated.tsv"
    }
REGIONS_FILE = "../cnv_inference/infercnv_runs/combined_stroma_ref/HMM_CNV_predictions.HMMi6.leiden.hmm_mode-subclusters.Pnorm_0.5.pred_cnv_regions.dat"
GROUPINGS_FILE = "../cnv_inference/infercnv_runs/combined_stroma_ref/infercnv.17_HMM_predHMMi6.leiden.hmm_mode-subclusters.observation_groupings.txt"

PFI_CAT_COL = "PFI"
SAMPLE_COL  = "patient"

PFI_ORDER   = ["short", "medium", "long"]
PFI_PALETTE = {'short': '#C7844A', 'medium': '#456EAE', 'long': '#538984'}

MIN_SPOTS_PER_SAMPLE = 10   # samples with fewer spots are excluded
ALPHA                = 0.05

OUT_DIR = "../outputs/associations"
import os; os.makedirs(OUT_DIR, exist_ok=True)

CHR19_LENGTH    = 58_617_616   # GRCh38 chr19 total length in bp
MIN_GAIN_STATE  = 4

MIN_SPOTS_PER_SAMPLE = 10
ALPHA                = 0.05

OUT_DIR = "../outputs/chr19_associations"
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from itertools   import combinations
from scipy.stats import kruskal, mannwhitneyu, spearmanr
from statsmodels.stats.multitest          import multipletests
from statsmodels.miscmodels.ordinal_model import OrderedModel
import anndata as ad

### Load inferCNV files and extract chr19 regions

We read the `pred_cnv_regions.dat` file and filter to chromosome 19.
The subcluster name is the part of `cell_group_name` after the first `.`.


In [ ]:
regions = pd.read_csv(REGIONS_FILE, sep="\t")
regions["subcluster"] = regions["cell_group_name"].str.split(".", n=1).str[-1]

chr19 = regions[regions["chr"] == "chr19"].copy()
chr19["region_bp"] = chr19["end"] - chr19["start"]

print(f"Total region rows          : {len(regions):,}")
print(f"Chr19 region rows          : {len(chr19):,}")
print(f"Unique chr19 region names  : {chr19['cnv_name'].nunique():,}")
print(f"Subclusters with chr19 data: {chr19['subcluster'].nunique()}")
print()
print("Chr19 HMM state distribution:")
print(chr19["state"].value_counts().sort_index().rename(
    {1:"1 complete loss", 2:"2 loss", 3:"3 neutral",
     4:"4 gain", 5:"5 amp", 6:"6 high-amp"}))

groupings = pd.read_csv(GROUPINGS_FILE, sep=" ", quotechar='"', index_col=0)
groupings.index.name = "barcode"
groupings = groupings.reset_index().rename(
    columns={"Dendrogram Group": "subcluster",
             "Annotation Group": "annotation_group"})
print(f"\nTotal spots in groupings: {len(groupings):,}")

### Compute per-subcluster chr19 gain metrics

For each subcluster we compute:
- **gain_bp**: total base pairs of chr19 covered by a gain region (state ≥ `MIN_GAIN_STATE`)
- **gain_frac**: gain_bp / CHR19_LENGTH — what fraction of chr19 is amplified
- **max_state**: the highest HMM state seen on chr19 for this subcluster
- **has_gain**: boolean — any gain at all


In [ ]:
# Gain rows only
chr19_gain = chr19[chr19["state"] >= MIN_GAIN_STATE].copy()

# Aggregate to subcluster level
gain_per_sub = (
    chr19_gain
    .assign(gain_bp=chr19_gain["end"] - chr19_gain["start"])
    .groupby("subcluster")
    .agg(gain_bp=("gain_bp", "sum"), max_state=("state", "max"))
    .reset_index()
)
gain_per_sub["gain_frac"] = gain_per_sub["gain_bp"] / CHR19_LENGTH

# Merge onto ALL subclusters in groupings (missing = neutral/no data -> fill with 0/3)
all_subs = pd.DataFrame({"subcluster": groupings["subcluster"].unique()})
all_subs  = all_subs.merge(gain_per_sub, on="subcluster", how="left")
all_subs["gain_bp"]   = all_subs["gain_bp"].fillna(0).astype(int)
all_subs["gain_frac"] = all_subs["gain_frac"].fillna(0.0)
all_subs["max_state"] = all_subs["max_state"].fillna(3).astype(int)   # 3 = neutral
all_subs["has_gain"]  = all_subs["gain_bp"] > 0

print("Chr19 gain per subcluster:")
print(f"  Subclusters with gain  : {all_subs['has_gain'].sum()} / {len(all_subs)}")
print(f"  Gain fraction range    : "
      f"{all_subs['gain_frac'].min():.3f} – {all_subs['gain_frac'].max():.3f}")
print(f"  Max state range        : "
      f"{all_subs['max_state'].min()} – {all_subs['max_state'].max()}")
print()
print(all_subs.sort_values("gain_frac", ascending=False).head(10).to_string(index=False))

## Map to spot barcodes and aggregate to sample level

In [ ]:
# Join metrics onto every spot
spot_df = groupings.merge(all_subs, on="subcluster", how="left")
spot_df["patient"] = spot_df["subcluster"].str.extract(r"^(H\d+)")

# Per-sample chr19 gain summary
sample_chr19 = spot_df.groupby("patient").agg(
    n_spots          = ("barcode",    "count"),
    n_spots_gained   = ("has_gain",   "sum"),
    mean_gain_frac   = ("gain_frac",  "mean"),   # mean gain fraction across spots
    mean_max_state   = ("max_state",  "mean"),   # mean severity across spots
).reset_index()

sample_chr19["spot_frac_gained"] = (
    sample_chr19["n_spots_gained"] / sample_chr19["n_spots"]
)

print("Per-patient chr19 gain metrics:")
print(sample_chr19.round(3).to_string(index=False))

# Save amplified spot barcodes for spatial plotting
gained_barcodes = spot_df[spot_df["has_gain"]]["barcode"]
bc_path = f"{OUT_DIR}/chr19_gain_spot_barcodes.txt"
gained_barcodes.to_csv(bc_path, index=False, header=False)
print(f"\nGained spot barcodes saved: {bc_path}  ({len(gained_barcodes):,} spots)")


### Load PFI categories and merge with chr19 metrics

In [ ]:
adata = ad.read_h5ad(ADATA_PATH)
adata = adata[~adata.obs['patient'].isin(['Marker'])].copy()
obs   = adata.obs[[PFI_CAT_COL]].copy()
if SAMPLE_COL and SAMPLE_COL in adata.obs.columns:
    obs["patient"] = adata.obs[SAMPLE_COL].astype(str)
else:
    obs["patient"] = adata.obs.index.str.extract(r"-(\d+)$")[0]
meta_df = (
    obs.groupby("patient")
    .agg(PFI_cat=(PFI_CAT_COL, "first"), n_spots=("patient","count"))
    .reset_index()
)
print(f"Loaded {len(meta_df)} patients from {ADATA_PATH}")

meta_df["PFI_cat"] = (
    meta_df["PFI_cat"].str.lower().str.strip()
    .pipe(lambda s: pd.Categorical(s, categories=PFI_ORDER, ordered=True))
)
meta_df["PFI_num"] = meta_df["PFI_cat"].cat.codes   # 0=short, 1=medium, 2=long

# Merge with chr19 metrics (keep n_spots from meta_df as the authoritative count)
analysis_df = meta_df.merge(
    sample_chr19.drop(columns="n_spots"), on="patient", how="left"
)
analysis_df = analysis_df[analysis_df["n_spots"] >= MIN_SPOTS_PER_SAMPLE]

print()
print("Analysis table:")
display_cols = ["patient","PFI_cat","n_spots","spot_frac_gained",
                "mean_gain_frac","mean_max_state"]
print(analysis_df[display_cols].round(3).to_string(index=False))
print()
print("PFI distribution:", dict(analysis_df["PFI_cat"].value_counts()[PFI_ORDER]))

### Overview plots

In [ ]:
metrics_info = {
    "spot_frac_gained" : "Fraction of spots with chr19 gain",
    "mean_gain_frac"   : "Mean chr19 bp fraction gained (per spot)",
    "mean_max_state"   : "Mean max HMM state on chr19 (per spot)",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (col, ylabel) in zip(axes, metrics_info.items()):
    df_s = analysis_df.sort_values(col, ascending=False)
    colors = [PFI_PALETTE[str(g)] for g in df_s["PFI_cat"]]
    ax.bar(df_s["patient"], df_s[col], color=colors,
           edgecolor="white", linewidth=0.5, zorder=2)
    ax.set_title(ylabel, fontsize=10, fontweight="bold")
    ax.set_xlabel("Patient"); ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=45)
    if col != "mean_max_state":
        ax.set_ylim(0, 1.05)
    else:
        ax.axhline(3, color="gray", linewidth=0.8, linestyle=":", alpha=0.6,
                   label="neutral (state 3)")
        ax.set_ylim(1, 6.5)
        ax.legend(fontsize=8)

legend_patches = [
    mpatches.Patch(facecolor=PFI_PALETTE[c], label=f"PFI: {c}")
    for c in PFI_ORDER
]
fig.legend(handles=legend_patches, loc="upper center",
           bbox_to_anchor=(.6, .8), fontsize=9, title="PFI category")
fig.suptitle("Chr19 gain burden per patient", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chr19_sample_overview.pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR}/chr19_sample_overview.pdf")

### Box + strip plots: chr19 metrics per PFI category

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (col, ylabel) in zip(axes, metrics_info.items()):
    sns.boxplot(data=analysis_df, x="PFI_cat", y=col,
                order=PFI_ORDER, palette=PFI_PALETTE,
                width=0.5, ax=ax, boxprops=dict(alpha=0.65), fliersize=0)
    sns.stripplot(data=analysis_df, x="PFI_cat", y=col,
                  order=PFI_ORDER, palette=PFI_PALETTE,
                  size=9, jitter=True, ax=ax,
                  edgecolor="white", linewidth=0.5, zorder=3)
    for _, row in analysis_df.iterrows():
        xpos = PFI_ORDER.index(str(row["PFI_cat"]))
        ax.annotate(row["patient"], xy=(xpos, row[col]),
                    xytext=(6, 0), textcoords="offset points",
                    fontsize=7, color="#444444", va="center")
    ax.set_title(ylabel, fontsize=10, fontweight="bold")
    ax.set_xlabel("PFI category"); ax.set_ylabel(ylabel)
    if col != "mean_max_state":
        ax.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chr19_boxplots.pdf", bbox_inches="tight")
plt.show()

### Statistical tests

We test all three chr19 metrics against PFI category using:
1. **Kruskal–Wallis** — global difference across 3 groups
2. **Jonckheere–Terpstra** — monotone trend (short > medium > long, one-sided)
3. **Pairwise Mann–Whitney + Bonferroni** — which pairs differ


In [ ]:
def jonckheere_terpstra(groups):
    """JT test for ordered alternatives. Returns (JT, z, p_one_sided)."""
    from scipy.stats import norm
    U_sum = E_sum = V_sum = 0
    for i in range(len(groups) - 1):
        for j in range(i + 1, len(groups)):
            ni, nj = len(groups[i]), len(groups[j])
            if ni == 0 or nj == 0: continue
            u, _ = mannwhitneyu(groups[i], groups[j], alternative="greater")
            U_sum += u
            E_sum += ni * nj / 2
            V_sum += ni * nj * (ni + nj + 1) / 12
    if V_sum == 0: return U_sum, np.nan, np.nan
    z = (U_sum - E_sum) / np.sqrt(V_sum)
    return U_sum, z, norm.sf(z)


all_stat_results = []   # collected for summary table

for col, ylabel in metrics_info.items():
    print(f"\n{'='*55}")
    print(f"  Metric: {ylabel}")
    print(f"{'='*55}")

    groups     = [analysis_df.loc[analysis_df["PFI_cat"]==c, col].values
                  for c in PFI_ORDER]
    valid_cats = [c for c, g in zip(PFI_ORDER, groups) if len(g) >= 2]
    valid_grps = [g for g in groups if len(g) >= 2]

    # Medians per category
    for cat, grp in zip(valid_cats, valid_grps):
        print(f"  {cat:<8}  n={len(grp)}  median={np.median(grp):.3f}")

    # Kruskal-Wallis
    kw_stat, kw_p = kruskal(*valid_grps) if len(valid_grps) >= 2 else (np.nan, np.nan)
    eta_sq = max(0, (kw_stat - len(valid_grps) + 1) / (len(analysis_df) - len(valid_grps)))
    kw_sig = " *" if kw_p < ALPHA else (" (trend)" if kw_p < 0.10 else "")
    print(f"\n  Kruskal-Wallis  H={kw_stat:.3f}  p={kw_p:.4f}{kw_sig}  eta2={eta_sq:.3f}")

    # Jonckheere-Terpstra
    jt_stat, jt_z, jt_p = jonckheere_terpstra(groups)
    jt_sig = " *" if jt_p < ALPHA else (" (trend)" if jt_p < 0.10 else "")
    print(f"  JT trend test   JT={jt_stat:.1f}  z={jt_z:.3f}  p={jt_p:.4f}{jt_sig}  (one-sided)")

    # Pairwise Mann-Whitney
    print(f"\n  Pairwise Mann-Whitney (Bonferroni-corrected, 3 comparisons):")
    pw_ps = []
    pw_rows = []
    for cat_a, cat_b in combinations(PFI_ORDER, 2):
        va = analysis_df.loc[analysis_df["PFI_cat"]==cat_a, col].values
        vb = analysis_df.loc[analysis_df["PFI_cat"]==cat_b, col].values
        if len(va) < 2 or len(vb) < 2:
            continue
        _, p2 = mannwhitneyu(va, vb, alternative="two-sided")
        _, p1 = mannwhitneyu(va, vb, alternative="greater")
        pw_ps.append(p2)
        pw_rows.append((cat_a, cat_b, np.median(va), np.median(vb), p2, p1))

    bonf = [min(p * len(pw_rows), 1.0) for p in pw_ps]
    for (ca, cb, ma, mb, p2, p1), pb in zip(pw_rows, bonf):
        sig = " *" if pb < ALPHA else ""
        print(f"    {ca:<8} vs {cb:<8}  "
              f"med {ma:.3f} vs {mb:.3f}  "
              f"p={p2:.4f}  p_bonf={pb:.4f}{sig}")

    all_stat_results.append({
        "metric"   : col,
        "kw_p"     : kw_p,
        "kw_eta2"  : eta_sq,
        "jt_p"     : jt_p,
        "jt_z"     : jt_z,
    })

print()


### Ordinal logistic regression

One model per chr19 metric. The odds ratio represents the change in odds of
being in a *higher* PFI category (better prognosis) per unit increase in the
chr19 gain metric.

- **OR < 1**: higher chr19 gain -> lower odds of better PFI -> worse prognosis ✓


In [ ]:
ordinal_results = []

for col, ylabel in metrics_info.items():
    fit_df = analysis_df[["PFI_cat", col]].dropna()
    if len(fit_df) < 5:
        print(f"  {col}: too few samples — skipping")
        continue
    try:
        mod = OrderedModel(fit_df["PFI_cat"], fit_df[[col]], distr="logit")
        res = mod.fit(method="bfgs", disp=False)
        coef  = res.params[col]
        se    = res.bse[col]
        p     = res.pvalues[col]
        OR    = np.exp(coef)
        ci_lo = np.exp(coef - 1.96 * se)
        ci_hi = np.exp(coef + 1.96 * se)
        ordinal_results.append({
            "metric"  : col,
            "label"   : ylabel,
            "OR"      : round(OR, 4),
            "CI_lo"   : round(ci_lo, 4),
            "CI_hi"   : round(ci_hi, 4),
            "p_value" : round(p, 4),
            "n"       : len(fit_df),
        })
        sig = " *" if p < ALPHA else (" (trend)" if p < 0.10 else "")
        print(f"  {col:<22}  OR={OR:.4f} [{ci_lo:.4f}-{ci_hi:.4f}]  "
              f"p={p:.4f}{sig}")
    except Exception as e:
        print(f"  {col}: ordinal model failed — {e}")

if ordinal_results:
    ord_df = pd.DataFrame(ordinal_results)
    _, ord_df["p_adj_BH"], _, _ = multipletests(ord_df["p_value"], method="fdr_bh")
    ord_df.to_csv(f"{OUT_DIR}/chr19_ordinal_regression.csv", index=False)
    print(f"\nSaved: {OUT_DIR}/chr19_ordinal_regression.csv")


### Forest plot of odds ratios

In [ ]:
if ordinal_results:
    df_fp = pd.DataFrame(ordinal_results)
    fig, ax = plt.subplots(figsize=(9, 3))
    y_pos = list(range(len(df_fp)))
    dot_colors = [PFI_PALETTE["short"] if p < ALPHA else "#95A5A6"
                  for p in df_fp["p_value"]]

    for i, (_, row) in enumerate(df_fp.iterrows()):
        ax.plot([row["CI_lo"], row["CI_hi"]], [i, i],
                color=dot_colors[i], linewidth=2.5,
                solid_capstyle="round", zorder=2)
        ax.scatter(row["OR"], i, color=dot_colors[i], s=90, zorder=3)

    ax.axvline(1.0, color="black", linewidth=1.2, linestyle="--", alpha=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_fp["label"].tolist(), fontsize=10)
    ax.set_xlabel("Odds Ratio (95% CI)\nOR < 1: higher chr19 gain -> worse PFI", fontsize=10)
    ax.set_title("Ordinal logistic regression\nChr19 gain metrics vs PFI category",
                 fontsize=12, fontweight="bold")
    x_right = df_fp["CI_hi"].max()
    for i, (_, row) in enumerate(df_fp.iterrows()):
        ax.text(x_right * 1.05, i,
                f"OR {row['OR']:.4f}  p={row['p_value']:.4f}",
                va="center", fontsize=9)
    ax.set_xlim(left=0)
    ax.legend(handles=[
        mpatches.Patch(facecolor=PFI_PALETTE["short"], label=f"p < {ALPHA}"),
        mpatches.Patch(facecolor="#95A5A6", label=f"p >= {ALPHA}"),
    ], fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/chr19_03_forest_plot.pdf", bbox_inches="tight")
    plt.show()
    print(f"Saved: {OUT_DIR}/chr19_03_forest_plot.pdf")


### Compare chr19 amplicon signal with individual gene signals

If the chr19-wide metrics are more significantly associated with PFI than individual BST2 or CCNE1 fractions, it supports the amplicon-as-driver hypothesis.
We load the per-gene annotated TSV files produced by `extract_cnv_spots.py` and compare Spearman correlations and KW p-values side by side.

In [ ]:
gene_files = {
    "BST2 amp"  : "../outputs/BST2_results/BST2_amplification_spots_annotated.tsv",
}

comparison_rows = []

# Add chr19 metrics
for col, ylabel in metrics_info.items():
    rho, rho_p = spearmanr(analysis_df[col], analysis_df["PFI_num"])
    groups = [analysis_df.loc[analysis_df["PFI_cat"]==c, col].values
              for c in PFI_ORDER]
    valid  = [g for g in groups if len(g) >= 2]
    kw_p = kruskal(*valid)[1] if len(valid) >= 2 else np.nan
    _, jt_z, jt_p = jonckheere_terpstra(groups)
    comparison_rows.append({
        "feature"   : f"Chr19 {col.replace('_',' ')}",
        "type"      : "chr19 amplicon",
        "spearman_rho": round(rho, 3),
        "spearman_p"  : round(rho_p, 4),
        "KW_p"        : round(kw_p, 4),
        "JT_p"        : round(jt_p, 4),
    })

# Add per-gene metrics
for gene_label, path in gene_files.items():
    if not os.path.exists(path):
        print(f"  Skipping {gene_label}: {path} not found")
        continue
    spots = pd.read_csv(path, sep="\t")
    spots["patient"] = spots["subcluster"].str.extract(r"^(H\d+)")
    cnv_counts = spots.groupby("patient").size().reset_index(name="n_cnv_spots")
    gene_df = analysis_df.merge(cnv_counts, on="patient", how="left")
    gene_df["n_cnv_spots"] = gene_df["n_cnv_spots"].fillna(0)
    gene_df["gene_frac"]   = gene_df["n_cnv_spots"] / gene_df["n_spots"]

    rho, rho_p = spearmanr(gene_df["gene_frac"], gene_df["PFI_num"])
    groups = [gene_df.loc[gene_df["PFI_cat"]==c, "gene_frac"].values
              for c in PFI_ORDER]
    valid  = [g for g in groups if len(g) >= 2]
    kw_p = kruskal(*valid)[1] if len(valid) >= 2 else np.nan
    _, jt_z, jt_p = jonckheere_terpstra(groups)
    comparison_rows.append({
        "feature"      : gene_label,
        "type"         : "per-gene CNV fraction",
        "spearman_rho" : round(rho, 3),
        "spearman_p"   : round(rho_p, 4),
        "KW_p"         : round(kw_p, 4),
        "JT_p"         : round(jt_p, 4),
    })

comp_df = pd.DataFrame(comparison_rows)
print("Chr19 amplicon vs individual gene metrics comparison:")
print(comp_df.to_string(index=False))
comp_df.to_csv(f"{OUT_DIR}/chr19_vs_gene_comparison.csv", index=False)